In [ ]:
# General library imports.
import sys
from pathlib import Path

# Find the example helpers from either the repository root or this directory.
notebook_dir = Path.cwd() if (Path.cwd() / "text_abstention.py").exists() else Path.cwd() / "quickstart_ipynbs"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

%load_ext autoreload
%autoreload 2
%aimport -torch -sklearn -numpy -np -pandas -tensorflow -transformers -trl

import pandas as pd
from IPython.display import display
from text_abstention import FLAG_QUESTIONS, REASON_FLAGS, text_abstained, summarize_predictions

In [ ]:
# CPU example: these responses and extraction flags are hand-written teaching cases.
# They are not model results or benchmark measurements.
# Paper: https://arxiv.org/abs/2603.27033 (Section 6.2).
# Text abstention in the evaluation: no species prediction AND at least one
# reason/ambiguity flag. A wrong species guess is not an abstention.
# This differs from thresholding an encoder's maximum class probability (Section 4).

def flags(species="No", *positive_reasons):
    result = {key: "No" for key in FLAG_QUESTIONS}
    result["is_species_predicted"] = species
    for key in positive_reasons:
        if key not in REASON_FLAGS:
            raise ValueError(key)
        result[key] = "Yes"
    return result

examples = [
    ("Clear identification", "This is Corvus corax.", flags("Yes")),
    ("Wrong confident guess", "This is Corvus corone.", flags("Yes")),
    ("Needs sound", "These species look alike; I need its call to identify it.", flags("No", "is_vocalization_needed")),
    ("Reasonless refusal", "I cannot identify it.", flags()),
    ("Genus only", "Corvus, but the diagnostic features are hidden.", flags("No", "is_occlusion_failure")),
    ("Reason with a species guess", "The image is blurry, but it is Corvus corax.", flags("Yes", "is_quality_failure")),
    ("Ambiguous prediction", "Either of two species is possible; I cannot choose.", flags("No", "is_prediction_ambiguous")),
]
case_table = pd.DataFrame([
    {"case": name, "response": response,
     "no_species_prediction": extracted["is_species_predicted"] == "No",
     "has_reason_or_ambiguity": any(extracted[key] == "Yes" for key in REASON_FLAGS),
     "abstained": text_abstained(extracted)}
    for name, response, extracted in examples
])
display(case_table)
assert case_table["abstained"].tolist() == [False, False, True, False, True, False, True]

In [ ]:
# The boolean rule does not decide whether the reason is correct.
# Inspect all flags; extraction errors must not silently become abstentions.
display(pd.DataFrame([{"flag": key, "question": value} for key, value in FLAG_QUESTIONS.items()]))
# is_prediction_ambiguous uses the same question with target answer "No".
# The other flags use target answer "Yes" in the original verification protocol.
# The end-to-end notebook demonstrates text-only constrained extraction.

In [ ]:
# Species accuracy uses answerable (A) rows only. UA has no species ground truth.
# Abstentions count as incorrect for unconditional A species accuracy.
# Selective accuracy uses only the non-abstained A rows.
records = [
    {"subset": "answerable", "ground_truth": "Corvus corax", "prediction": "Corvus corax", "abstained": False},
    {"subset": "answerable", "ground_truth": "Corvus corax", "prediction": "Corvus corone", "abstained": False},
    {"subset": "answerable", "ground_truth": "Corvus corax", "prediction": None, "abstained": True},
    {"subset": "unanswerable", "ground_truth": "Corvus", "prediction": None, "abstained": True},
    {"subset": "unanswerable", "ground_truth": "Corvus", "prediction": "Corvus corax", "abstained": False},
]
metrics = summarize_predictions(records)
display(pd.Series(metrics, name="synthetic demonstration"))
assert metrics["species_accuracy_answerable"] == 1 / 3
assert metrics["selective_species_accuracy_answerable"] == 1 / 2
assert metrics["abstention_rate_all"] == 2 / 5
assert metrics["abstention_rate_unanswerable"] == 1 / 2

In [ ]:
# Edge case: selective accuracy is undefined if no answerable rows are answered.
all_abstained = [{**row, "prediction": None, "abstained": True} for row in records]
edge_metrics = summarize_predictions(all_abstained)
assert pd.isna(edge_metrics["selective_species_accuracy_answerable"])
display(pd.Series(edge_metrics, name="all abstained"))
# Keep prompt wording and extraction checkpoint fixed when comparing models.
# Do not interpret this rate as the paper's threshold-swept UA/A or IG metric.